In [1]:
import torch.nn as nn
import torch
import torchvision.transforms.functional as functional


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False), #kernal_size, stride, padding
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class UNET2D(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features = [64, 128, 256, 512]):
        super(UNET2D, self).__init__()
        self.downsamplings = nn.ModuleList()
        self.upsamplings = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        #down sampling
        for feature in features:
            self.downsamplings.append(DoubleConv(in_channels, feature))
            in_channels = feature

        #up sampling
        for feature in reversed(features):
            self.upsamplings.append(nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    DoubleConv(feature * 2, feature)  # This ensures the channel count is correct after concatenation
                )
            )
            self.upsamplings.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        #skip connection
        skip_connection = []

        for down_sampling in self.downsamplings:
            x = down_sampling(x)
            skip_connection.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connection = skip_connection[::-1] #reverse

        for idx  in range(0, len(self.upsamplings), 2):
            x = self.upsamplings[idx](x)
            skip_connection_feature = skip_connection[idx//2]
            concatenated_features = torch.cat([x, skip_connection_feature], dim=1)
            x = self.upsamplings[idx+1](concatenated_features)


        return self.final_conv(x)

ex = torch.rand((3, 3, 160, 160))
model = UNET(in_channels=3, out_channels=1)
out = model(ex)
print(out.shape)


NameError: name 'UNET' is not defined

In [ ]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super(ExampleModel, self).__init__()
        feature = 64
        self.upsamplings = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(feature * 2, feature, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x):
        for layer in self.upsamplings:
            x = layer(x)
            print(f'Layer: {layer.__class__.__name__}, Output size: {x.size()}')
        return x


class DoubleConv(nn.Module):
    def __init__(self):
        super(DoubleConv, self).__init__()

    def forward(self, x):
        feature = 64
        layer = nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
        x = layer(x)
        print(f'Layer: {layer.__class__.__name__}, Output size: {x.size()}')
        return x

# Create an instance of the model
model = ExampleModel()
model2 = DoubleConv()
# Create a sample input tensor with batch size 1, 128 channels, and 16x16 spatial dimensions
input_tensor = torch.randn(1, 128, 16, 16)

# Pass the input tensor through the model
output = model(input_tensor)
print("---------")
output2 = model2(input_tensor)

Layer: Upsample, Output size: torch.Size([1, 128, 32, 32])
Layer: Conv2d, Output size: torch.Size([1, 64, 32, 32])
---------
Layer: ConvTranspose2d, Output size: torch.Size([1, 64, 32, 32])


# Rwightman (timm)

In [ ]:
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.5 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many